# 18 - Final Inverse Design Validation

## Objective

This notebook prepares and validates the final inverse-design submission.

The inverse-design file is:

```text
outputs/submissions/design_submission.csv
```

The final forward decision is now based on the hybrid MLP-distance model.  
Therefore, the inverse-design scenarios must remain feasible not only under the conservative baseline, but also under the final hybrid forward candidate.

This notebook is a final validation notebook. It does not search for new designs. It checks that the existing 20 designs are valid, inside input bounds, and robust under the forward recheck diagnostics.

Final decision:

```text
Forward file: prediction_submission.csv
Inverse-design file: design_submission.csv
```

Recommended inverse-design source:

```text
outputs/submissions/design_submission.csv
```

## 1. Imports and project paths

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

INVERSE_DIR = PROJECT_ROOT / "data" / "raw" / "inverse_design"
SUBMISSIONS_DIR = PROJECT_ROOT / "outputs" / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"

print("Project root:", PROJECT_ROOT)
print("Inverse data:", INVERSE_DIR)
print("Submissions:", SUBMISSIONS_DIR)
print("Reports:", REPORTS_DIR)

Project root: /home/alouiyaz/projects/boom-challenge-ejecta-prediction
Inverse data: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/data/raw/inverse_design
Submissions: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions
Reports: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/reports


## 2. Load constraints and final inverse-design file

In [2]:
input_cols = [
    "energy", "angle_rad", "coupling", "strength",
    "porosity", "gravity", "atmosphere", "shape_factor",
]

expected_design_cols = ["submission_id"] + input_cols

constraints_path = INVERSE_DIR / "constraints.json"

if not constraints_path.exists():
    raise FileNotFoundError(
        f"Missing constraints file: {constraints_path}. "
        "Please check the inverse_design raw data folder."
    )

with open(constraints_path, "r") as f:
    constraints_data = json.load(f)

output_constraints = constraints_data["constraints"]
input_bounds = constraints_data["input_bounds"]

p80_min = output_constraints["p80_min"]
p80_max = output_constraints["p80_max"]
r95_max = output_constraints["r95_max"]

design_path = SUBMISSIONS_DIR / "design_submission.csv"

if not design_path.exists():
    raise FileNotFoundError(f"Missing final inverse-design file: {design_path}")

design_submission = pd.read_csv(design_path)

print("Design submission:", design_submission.shape)
print("Output constraints:")
display(pd.DataFrame([output_constraints]))

print("Input bounds:")
display(pd.DataFrame(input_bounds).T)

display(design_submission.head())

Design submission: (20, 9)
Output constraints:


,p80_min,p80_max,r95_max
0,96.0,101.0,175.0


Input bounds:


,min,max
energy,0.500000,5.000000
angle_rad,0.261799,1.570796
coupling,0.200000,1.700000
strength,0.400000,4.200000
porosity,0.000000,0.330000
gravity,1.020000,10.470000
atmosphere,0.000000,1.000000
shape_factor,0.700000,1.500000


,submission_id,energy,angle_rad,coupling,strength,porosity,gravity,atmosphere,shape_factor
0,0,3.598016,0.797647,1.291533,2.024534,0.304489,9.851536,0.422767,1.303159
1,1,3.219857,0.653336,0.937745,1.371766,0.268351,10.031512,0.819009,0.775107
2,2,3.883410,0.550208,0.754591,1.428060,0.294039,9.893283,0.800587,0.798041
3,3,2.769840,0.856038,0.933836,1.186456,0.266355,9.856101,0.429999,1.214199
4,4,3.491508,0.545414,1.620353,2.249113,0.264882,9.960773,0.715197,0.816463


## 3. Format and input-bound checks

In [3]:
def validate_design_format(df):
    assert df.shape == (20, 9), f"Expected design shape (20, 9), got {df.shape}"
    assert df.columns.tolist() == expected_design_cols, "Unexpected design columns"
    assert df["submission_id"].tolist() == list(range(20)), "submission_id must be 0..19"
    assert np.isfinite(df.drop(columns=["submission_id"]).values).all(), "Design contains non-finite values"

    print("Design format OK")


def validate_input_bounds(df):
    rows = []

    for col in input_cols:
        lower = input_bounds[col]["min"]
        upper = input_bounds[col]["max"]

        min_value = df[col].min()
        max_value = df[col].max()

        inside = bool((df[col].between(lower, upper)).all())

        rows.append({
            "input": col,
            "bound_min": lower,
            "observed_min": min_value,
            "observed_max": max_value,
            "bound_max": upper,
            "inside_bounds": inside,
            "lower_margin": min_value - lower,
            "upper_margin": upper - max_value,
        })

    bounds_check = pd.DataFrame(rows)

    assert bounds_check["inside_bounds"].all(), "At least one design variable is outside allowed bounds"

    print("Input bounds OK")
    return bounds_check


validate_design_format(design_submission)
bounds_check = validate_input_bounds(design_submission)
display(bounds_check)

Design format OK
Input bounds OK


,input,bound_min,observed_min,observed_max,bound_max,inside_bounds,lower_margin,upper_margin
0,energy,0.500000,2.769840,4.111200,5.000000,True,2.269840,0.888800
1,angle_rad,0.261799,0.501950,0.856038,1.570796,True,0.240151,0.714758
2,coupling,0.200000,0.752039,1.620353,1.700000,True,0.552039,0.079647
3,strength,0.400000,1.143469,2.416876,4.200000,True,0.743469,1.783124
4,porosity,0.000000,0.251777,0.306724,0.330000,True,0.251777,0.023276
5,gravity,1.020000,9.309159,10.083053,10.470000,True,8.289159,0.386947
6,atmosphere,0.000000,0.422767,0.864865,1.000000,True,0.422767,0.135135
7,shape_factor,0.700000,0.775107,1.333853,1.500000,True,0.075107,0.166147


## 4. Load inverse-design recheck diagnostics

The detailed feasibility recheck is produced by the previous inverse-design diagnostic notebook.

Expected files:

```text
reports/inverse_design_recheck_with_distance_blends.csv
reports/inverse_design_recheck_summary.csv
```

These diagnostics compare the current 20 designs under:

- base ExtraTrees;
- hybrid MLP-distance model;
- alpha25 controlled blend;
- alpha40 controlled blend.

Because the final forward model is now the hybrid candidate, the key check is:

```text
all 20 designs must remain feasible under hybrid predictions.
```

In [4]:
recheck_path = REPORTS_DIR / "inverse_design_recheck_with_distance_blends.csv"
summary_path = REPORTS_DIR / "inverse_design_recheck_summary.csv"

if not recheck_path.exists():
    raise FileNotFoundError(
        f"Missing detailed recheck file: {recheck_path}. "
        "Please run the inverse-design recheck notebook first."
    )

if not summary_path.exists():
    raise FileNotFoundError(
        f"Missing recheck summary file: {summary_path}. "
        "Please run the inverse-design recheck notebook first."
    )

recheck = pd.read_csv(recheck_path)
recheck_summary = pd.read_csv(summary_path)

display(recheck_summary)
display(recheck.head())

,model,n_feasible,n_total,all_feasible,P80_min,P80_mean,P80_max,R95_min,R95_mean,R95_max,R95_margin_min
0,base,20,20,True,97.362165,98.520519,99.709877,74.922776,85.931067,101.316388,73.683612
1,hybrid,20,20,True,97.362165,98.520519,99.709877,78.624967,84.734990,97.115946,77.884054
2,alpha25,20,20,True,97.362165,98.520519,99.709877,76.039612,85.632048,100.266277,74.733723
3,alpha40,20,20,True,97.362165,98.520519,99.709877,76.709713,85.452636,99.636211,75.363789


,submission_id,energy,angle_rad,coupling,strength,porosity,gravity,atmosphere,shape_factor,base_P80,base_fines_frac,base_oversize_frac,base_R95,base_R50_fines,base_R50_oversize,hybrid_P80,hybrid_fines_frac,hybrid_oversize_frac,hybrid_R95,hybrid_R50_fines,hybrid_R50_oversize,alpha25_P80,alpha25_fines_frac,alpha25_oversize_frac,alpha25_R95,alpha25_R50_fines,alpha25_R50_oversize,alpha40_P80,alpha40_fines_frac,alpha40_oversize_frac,alpha40_R95,alpha40_R50_fines,alpha40_R50_oversize,base_feasible,hybrid_feasible,alpha25_feasible,alpha40_feasible
0,0,3.598016,0.797647,1.291533,2.024534,0.304489,9.851536,0.422767,1.303159,98.663679,0.093206,0.072531,88.973302,86.308420,38.441772,98.663679,0.093206,0.072531,90.200049,85.412697,37.607795,98.663679,0.093206,0.072531,89.279989,86.084489,38.233278,98.663679,0.093206,0.072531,89.464001,85.950131,38.108182,True,True,True,True
1,1,3.219857,0.653336,0.937745,1.371766,0.268351,10.031512,0.819009,0.775107,98.648438,0.084552,0.072528,85.952720,83.312234,36.251774,98.648438,0.084552,0.072528,83.265891,80.145406,36.953548,98.648438,0.084552,0.072528,85.281013,82.520527,36.427217,98.648438,0.084552,0.072528,84.877988,82.045503,36.532483,True,True,True,True
2,2,3.883410,0.550208,0.754591,1.428060,0.294039,9.893283,0.800587,0.798041,99.442042,0.077125,0.074867,77.115577,73.694526,32.081176,99.442042,0.077125,0.074867,82.473864,77.600209,36.597609,99.442042,0.077125,0.074867,78.455149,74.670947,33.210284,99.442042,0.077125,0.074867,79.258892,75.256799,33.887749,True,True,True,True
3,3,2.769840,0.856038,0.933836,1.186456,0.266355,9.856101,0.429999,1.214199,97.907049,0.086700,0.062362,81.647131,81.902085,35.894790,97.907049,0.086700,0.062362,80.347309,83.964264,35.768614,97.907049,0.086700,0.062362,81.322175,82.417630,35.863246,97.907049,0.086700,0.062362,81.127202,82.726956,35.844320,True,True,True,True
4,4,3.491508,0.545414,1.620353,2.249113,0.264882,9.960773,0.715197,0.816463,97.735768,0.099837,0.071675,83.307290,76.578171,32.180152,97.735768,0.099837,0.071675,87.631624,76.223228,36.265539,97.735768,0.099837,0.071675,84.388374,76.489435,33.201499,97.735768,0.099837,0.071675,85.037024,76.436193,33.814307,True,True,True,True


## 5. Final feasibility checks under all forward variants

In [5]:
required_models = ["base", "hybrid", "alpha25", "alpha40"]

for model in required_models:
    feasible_col = f"{model}_feasible"
    assert feasible_col in recheck.columns, f"Missing feasibility column: {feasible_col}"

    n_feasible = int(recheck[feasible_col].sum())
    print(f"{model}: {n_feasible} / {len(recheck)} feasible")

    assert n_feasible == 20, f"{model}: not all inverse-design rows are feasible"

print("\nAll 20 inverse-design scenarios are feasible under all checked forward variants.")

base: 20 / 20 feasible
hybrid: 20 / 20 feasible
alpha25: 20 / 20 feasible
alpha40: 20 / 20 feasible

All 20 inverse-design scenarios are feasible under all checked forward variants.


## 6. Final hybrid-specific margins

Since the final forward submission uses the hybrid model, we inspect the hybrid margins directly.

The design is valid when:

```text
96 <= P80 <= 101
R95 <= 175
```

A larger positive margin means the design is safer with respect to the constraint.

In [6]:
hybrid_check = recheck[[
    "submission_id",
    "hybrid_P80",
    "hybrid_R95",
    "hybrid_feasible",
]].copy()

hybrid_check["P80_lower_margin"] = hybrid_check["hybrid_P80"] - p80_min
hybrid_check["P80_upper_margin"] = p80_max - hybrid_check["hybrid_P80"]
hybrid_check["R95_margin"] = r95_max - hybrid_check["hybrid_R95"]

display(hybrid_check)

hybrid_margin_summary = pd.DataFrame([{
    "n_designs": len(hybrid_check),
    "n_hybrid_feasible": int(hybrid_check["hybrid_feasible"].sum()),
    "hybrid_P80_min": hybrid_check["hybrid_P80"].min(),
    "hybrid_P80_mean": hybrid_check["hybrid_P80"].mean(),
    "hybrid_P80_max": hybrid_check["hybrid_P80"].max(),
    "hybrid_R95_min": hybrid_check["hybrid_R95"].min(),
    "hybrid_R95_mean": hybrid_check["hybrid_R95"].mean(),
    "hybrid_R95_max": hybrid_check["hybrid_R95"].max(),
    "min_P80_lower_margin": hybrid_check["P80_lower_margin"].min(),
    "min_P80_upper_margin": hybrid_check["P80_upper_margin"].min(),
    "min_R95_margin": hybrid_check["R95_margin"].min(),
}])

display(hybrid_margin_summary)

assert hybrid_margin_summary["n_hybrid_feasible"].iloc[0] == 20
assert hybrid_margin_summary["min_R95_margin"].iloc[0] > 0

hybrid_margin_summary.to_csv(REPORTS_DIR / "18_final_hybrid_inverse_design_margin_summary.csv", index=False)

,submission_id,hybrid_P80,hybrid_R95,hybrid_feasible,P80_lower_margin,P80_upper_margin,R95_margin
0,0,98.663679,90.200049,True,2.663679,2.336321,84.799951
1,1,98.648438,83.265891,True,2.648438,2.351562,91.734109
2,2,99.442042,82.473864,True,3.442042,1.557958,92.526136
3,3,97.907049,80.347309,True,1.907049,3.092951,94.652691
4,4,97.735768,87.631624,True,1.735768,3.264232,87.368376
5,5,97.931658,79.075659,True,1.931658,3.068342,95.924341
6,6,98.327340,79.390119,True,2.327340,2.672660,95.609881
7,7,98.666910,85.815183,True,2.666910,2.333090,89.184817
8,8,98.185973,78.624967,True,2.185973,2.814027,96.375033
9,9,98.665610,90.862435,True,2.665610,2.334390,84.137565


,n_designs,n_hybrid_feasible,hybrid_P80_min,hybrid_P80_mean,hybrid_P80_max,hybrid_R95_min,hybrid_R95_mean,hybrid_R95_max,min_P80_lower_margin,min_P80_upper_margin,min_R95_margin
0,20,20,97.362165,98.520519,99.709877,78.624967,84.73499,97.115946,1.362165,1.290123,77.884054


## 7. Final design diversity summary

This is not a hard requirement, but it helps show that the 20 selected scenarios are not exact duplicates.

In [7]:
design_diversity = design_submission[input_cols].describe().T
display(design_diversity)

design_diversity.to_csv(REPORTS_DIR / "18_final_inverse_design_input_distribution.csv")

,count,mean,std,min,25%,50%,75%,max
energy,20.0,3.388203,0.432321,2.769840,3.117975,3.355683,3.676497,4.111200
angle_rad,20.0,0.675099,0.101456,0.501950,0.579074,0.693846,0.742868,0.856038
coupling,20.0,1.088042,0.279169,0.752039,0.820889,1.046774,1.314625,1.620353
strength,20.0,1.618387,0.374418,1.143469,1.343432,1.516952,1.731167,2.416876
porosity,20.0,0.277723,0.019303,0.251777,0.264408,0.271333,0.293259,0.306724
gravity,20.0,9.947327,0.171099,9.309159,9.914394,9.962109,10.083053,10.083053
atmosphere,20.0,0.660597,0.149585,0.422767,0.512000,0.676104,0.802581,0.864865
shape_factor,20.0,1.057618,0.187216,0.775107,0.890069,1.046821,1.196302,1.333853


## 8. Final submission files validation

This cell validates the two files expected by the challenge:

```text
outputs/submissions/prediction_submission.csv
outputs/submissions/design_submission.csv
```

In [8]:
prediction_path = SUBMISSIONS_DIR / "prediction_submission.csv"

if not prediction_path.exists():
    raise FileNotFoundError(f"Missing final prediction file: {prediction_path}")

prediction_submission = pd.read_csv(prediction_path)

expected_prediction_cols = [
    "scenario_id", "P80", "fines_frac", "oversize_frac",
    "R95", "R50_fines", "R50_oversize",
]

assert prediction_submission.shape == (492, 7), f"Expected prediction shape (492, 7), got {prediction_submission.shape}"
assert prediction_submission.columns.tolist() == expected_prediction_cols, "Unexpected prediction columns"
assert prediction_submission["scenario_id"].tolist() == list(range(492)), "scenario_id must be 0..491"
assert np.isfinite(prediction_submission.drop(columns=["scenario_id"]).values).all(), "Prediction file contains non-finite values"

assert (prediction_submission[["P80", "R95", "R50_fines", "R50_oversize"]] >= 0).all().all()
assert ((prediction_submission[["fines_frac", "oversize_frac"]] >= 0) &
        (prediction_submission[["fines_frac", "oversize_frac"]] <= 1)).all().all()

validate_design_format(design_submission)

print("Final submission files OK.")
print("Prediction:", prediction_path)
print("Design:", design_path)

display(prediction_submission.head())
display(design_submission.head())

Design format OK
Final submission files OK.
Prediction: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission.csv
Design: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/design_submission.csv


,scenario_id,P80,fines_frac,oversize_frac,R95,R50_fines,R50_oversize
0,0,120.852192,0.120046,0.208373,1064.238127,1012.805657,448.529033
1,1,113.668032,0.048219,0.158274,180.340190,191.185816,88.486206
2,2,149.740914,0.121343,0.384067,1306.856464,1316.118523,541.277606
3,3,162.227452,0.008732,0.543184,731.377793,891.581710,418.480874
4,4,137.978607,0.047156,0.357337,152.512030,142.451662,60.233150


,submission_id,energy,angle_rad,coupling,strength,porosity,gravity,atmosphere,shape_factor
0,0,3.598016,0.797647,1.291533,2.024534,0.304489,9.851536,0.422767,1.303159
1,1,3.219857,0.653336,0.937745,1.371766,0.268351,10.031512,0.819009,0.775107
2,2,3.883410,0.550208,0.754591,1.428060,0.294039,9.893283,0.800587,0.798041
3,3,2.769840,0.856038,0.933836,1.186456,0.266355,9.856101,0.429999,1.214199
4,4,3.491508,0.545414,1.620353,2.249113,0.264882,9.960773,0.715197,0.816463


## 9. Final inverse-design decision

The existing inverse-design file is kept:

```text
outputs/submissions/design_submission.csv
```

Reason:

- The file has the required format: 20 rows and 9 columns.
- All input variables are within the challenge bounds.
- All 20 scenarios remain feasible under the final hybrid forward model.
- All 20 scenarios also remain feasible under base, alpha25, and alpha40 rechecks.
- The minimum R95 margin under the hybrid check remains positive.

This means the inverse-design submission does not need to be regenerated after selecting the hybrid forward file.